# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font>  — Avance 8</center>

## **<font color="#0036a3">Avance 8 — Cuadrículas: mejores enhancements × modelos de profundidad</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10 · Equipo 52</font>**

---

Genera **dos cuadrículas** (una por cada mejor método de realce: **IAT** y **Endo-LMSPEC**). Cada cuadrícula tiene **4 filas** de fotogramas — **2 sub-expuestos** y **2 sobre-expuestos**, todos con **cobertura de ground truth > 30%** — y las columnas:

`input (raw)` · `output (enhanced)` · y el **mapa de profundidad** predicho por cada uno de los **5 modelos** (Monodepth2, EndoSfMLearner, AF-SfMLearner, MonoViT, MonoIIT).

> Reutiliza el setup del Avance 5 (carga de los 5 modelos + enhancements + split oficial). Re-ejecutar las celdas de configuración una vez, luego la celda de las cuadrículas.

## 1. Configuración (setup del Avance 5: split, 5 modelos, enhancements)

In [ ]:
from pathlib import Path
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE          = Path("/content/drive/MyDrive/proyecto_integrador")
    SCARED_ROOT   = BASE / "scared_raw"
    EDAM_PATH     = BASE / "Endo-Depth-and-Motion"    # codigo Monodepth2
    LMSPEC_PATH   = BASE / "EndoLMSPEC"
    IAT_PATH      = BASE / "EndoViT"
    ENDOSLAM_PATH = BASE / "EndoSLAM"
    MONOVIT_PATH  = BASE / "MonoViT"
    AFSFM_PATH    = BASE / "AF-SfMLearner"
    STTN_PATH     = BASE / "Endo-STTN"

    W = BASE / "scared weights"
    W_MONO2   = W / "monodepth2_weights" / "weights_19"
    W_MONOVIT = W / "monovit_weights" / "weights_19"
    W_ENDOSFM = W / "endosfmlearner_weights" / "11-09-03_58"
    W_AFSFM   = W / "afmlearner_weights" / "Model_trained_end_to_end"
    W_MONOIIT = W / "monoIIT_weights" / "trained-winner-weights"
    W_W19MONO = W / "weights_19_MonoViT" / "weights_19"

    REPO_ROOT = Path("/content/repo_52")
    if not REPO_ROOT.exists():
        subprocess.check_call([
            "git", "clone", "--depth=1",
            "https://github.com/jmtoral/proyecto_integrador_52.git",
            str(REPO_ROOT)
        ])
    else:
        # repo ya clonado en una sesion previa: actualizar para traer data/splits, etc.
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "reset", "--hard", "origin/main"])
    subprocess.check_call(["git","-C",str(REPO_ROOT),"config","user.email","jmtoralcruz@gmail.com"])
    subprocess.check_call(["git","-C",str(REPO_ROOT),"config","user.name","jmtoral"])
    OUT_DIR    = REPO_ROOT / "outcomes" / "avance5_newversion"
    SPLIT_FILE = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
    FRAMES_CACHE = Path("/content/split_frames")     # frames+GT extraidos (efimero, rapido)

else:
    BASE          = Path("E:/scared_wights_complete/scared weights")
    SCARED_ROOT   = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH     = Path("E:/Endo-Depth-and-Motion")
    LMSPEC_PATH   = Path("E:/EndoLMSPEC")
    IAT_PATH      = Path("E:/EndoVit")
    ENDOSLAM_PATH = Path("E:/EndoSLAM")
    MONOVIT_PATH  = Path("E:/MonoViT")
    AFSFM_PATH    = Path("E:/AF-SfMLearner")
    STTN_PATH     = Path("E:/Endo-STTN")

    W = BASE
    W_MONO2   = W / "monodepth2_weights" / "weights_19"
    W_MONOVIT = W / "monovit_weights" / "weights_19"
    W_ENDOSFM = W / "endosfmlearner_weights" / "11-09-03_58"
    W_AFSFM   = W / "afmlearner_weights" / "Model_trained_end_to_end"
    W_MONOIIT = W / "monoIIT_weights" / "trained-winner-weights"
    W_W19MONO = W / "weights_19_MonoViT" / "weights_19"

    REPO_ROOT  = Path(r"d:\Proyecto_Integrador\Corrreccion_Luz")
    OUT_DIR    = REPO_ROOT / "outcomes" / "avance5_newversion"
    SPLIT_FILE = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
    FRAMES_CACHE = REPO_ROOT / "data" / "split_frames"

LMSPEC_WEIGHTS  = LMSPEC_PATH / "checkpoint" / "main_net" / "model_256_combined_SSIM5_1.pth"
IAT_WEIGHTS     = IAT_PATH / "Endo4IE" / "best_Epoch50_laplacian_histogan_loss.pth"
ENDOSFM_WEIGHTS = W_ENDOSFM / "dispnet_model_best.pth.tar"

# Pesos Endo-STTN (gen_00009.pth). En Colab se descargan de Drive si no existen.
STTN_CKPT_DIR    = STTN_PATH / "release_model" / "pretrained_model"
STTN_CKPT_NUMBER = "9"     # gen_00009.pth
STTN_WEIGHTS     = STTN_CKPT_DIR / "gen_00009.pth"
STTN_GDRIVE_ID   = "14sdaDejsxgRuzHBSuqH2xEpbxuqyWI-R"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_CACHE.mkdir(parents=True, exist_ok=True)
CAP_MM = 150.0

# ---- Split oficial AF-SfMLearner (endovis): 550 frames, datasets 1-7 ----
# Formato por linea:  "dataset3/keyframe4 390 l"
def load_split(split_file):
    items = []
    with open(split_file) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            folder, frame_id, _side = line.split()
            # folder = "dataset3/keyframe4" -> dataset_3, keyframe_4
            ds, kf = folder.split("/")
            ds_n = "dataset_" + ds.replace("dataset","")
            kf_n = "keyframe_" + kf.replace("keyframe","")
            items.append((ds_n, kf_n, int(frame_id)))
    return items

SPLIT_ITEMS = load_split(SPLIT_FILE)   # lista de (dataset_N, keyframe_M, frame_id)

print(f"Entorno : {'Colab' if IN_COLAB else 'Local'}")
print(f"OUT_DIR : {OUT_DIR}")
print(f"Split   : {len(SPLIT_ITEMS)} frames  (datasets {sorted({d for d,_,_ in SPLIT_ITEMS})})")
for n, p in [("SCARED_ROOT",SCARED_ROOT),("LMSPEC_WEIGHTS",LMSPEC_WEIGHTS),
             ("IAT_WEIGHTS",IAT_WEIGHTS),("ENDOSFM_WEIGHTS",ENDOSFM_WEIGHTS),
             ("W_MONO2",W_MONO2),("W_MONOVIT",W_MONOVIT),
             ("W_AFSFM",W_AFSFM),("W_MONOIIT",W_MONOIIT),
             ("W_W19MONO",W_W19MONO),("STTN_PATH",STTN_PATH)]:
    print(f"  {n:20s}: {'OK' if Path(p).exists() else 'NO ENCONTRADO'}")


In [ ]:
import io, zipfile, tarfile, cv2, numpy as np, tifffile
from collections import defaultdict
from tqdm import tqdm

# Caché persistente empaquetado en Drive (sobrevive entre sesiones de Colab)
NPZ_CACHE = BASE / "split_frames.npz"
def _key(ds, kf, fid): return f"{ds}|{kf}|{fid}"

def _frame_paths(ds, kf, fid):
    base = FRAMES_CACHE / ds / kf
    return base / f"img_{fid:010d}.png", base / f"gt_{fid:010d}.npy"

def extract_split_frames(split_items, scared_root, force=False):
    """Extrae imagen+GT de cada (ds,kf,frame). Scratch en FRAMES_CACHE (rapido)."""
    by_kf = defaultdict(list)
    for ds, kf, fid in split_items:
        by_kf[(ds, kf)].append(fid)
    n_done = 0
    for (ds, kf), fids in tqdm(by_kf.items(), desc="Keyframes del split"):
        fids = sorted(set(fids))
        pend = [f for f in fids if force or not _frame_paths(ds, kf, f)[0].exists()
                                        or not _frame_paths(ds, kf, f)[1].exists()]
        if not pend: continue
        (FRAMES_CACHE / ds / kf).mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(scared_root / f"{ds}.zip") as z:
            sp_bytes = z.read(f"{ds}/{kf}/data/scene_points.tar.gz")
            with tarfile.open(fileobj=io.BytesIO(sp_bytes)) as t:
                tnames = {n.split("/")[-1]: n for n in t.getnames() if n.endswith(".tiff")}
                gt_cache = {}
                for fid in pend:
                    key = f"scene_points{fid-1:06d}.tiff"
                    if key not in tnames: gt_cache[fid] = None; continue
                    raw = t.extractfile(tnames[key]).read()
                    tiff = tifffile.imread(io.BytesIO(raw))
                    dz = tiff[..., 2].astype(np.float32) if tiff.ndim == 3 else tiff.astype(np.float32)
                    dz = dz[0:1024, :]; dz[dz <= 0] = np.nan; gt_cache[fid] = dz
            tmp_mp4 = FRAMES_CACHE / "_tmp.mp4"
            with open(tmp_mp4, "wb") as fout: fout.write(z.read(f"{ds}/{kf}/data/rgb.mp4"))
            cap = cv2.VideoCapture(str(tmp_mp4)); want = set(pend); maxf = max(pend)
            idx = 0; got = {}
            while idx <= maxf:
                ok, frame = cap.read()
                if not ok: break
                if idx in want: got[idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)[0:1024, :]
                idx += 1
            cap.release(); tmp_mp4.unlink(missing_ok=True)
        for fid in pend:
            img_p, gt_p = _frame_paths(ds, kf, fid)
            if fid in got: cv2.imwrite(str(img_p), cv2.cvtColor(got[fid], cv2.COLOR_RGB2BGR))
            if gt_cache.get(fid) is not None: np.save(gt_p, gt_cache[fid])
            n_done += 1
    print(f"Extracción lista — {n_done} frames nuevos")

# Diccionario en memoria: clave -> (img_uint8, gt_float32|None)
SPLIT_DATA = {}

if NPZ_CACHE.exists() and not globals().get("FORCE_REEXTRACT", False):
    # ---- Carga rapida desde Drive (segundos) ----
    print(f"Cargando caché empaquetado: {NPZ_CACHE.name}")
    _npz = np.load(NPZ_CACHE, allow_pickle=True)
    for ds, kf, fid in SPLIT_ITEMS:
        k = _key(ds, kf, fid)
        ik, gk = "img_"+k, "gt_"+k
        if ik in _npz.files:
            gt = _npz[gk] if gk in _npz.files else None
            if gt is not None and gt.size == 1 and np.isnan(gt).all(): gt = None
            SPLIT_DATA[k] = (_npz[ik], gt)
    print(f"Caché cargado — {len(SPLIT_DATA)} frames listos (sin re-extraer)")
else:
    # ---- Primera vez: extraer y empaquetar en Drive ----
    extract_split_frames(SPLIT_ITEMS, SCARED_ROOT)
    save_dict = {}
    for ds, kf, fid in SPLIT_ITEMS:
        img_p, gt_p = _frame_paths(ds, kf, fid)
        if not img_p.exists(): continue
        k = _key(ds, kf, fid)
        img = cv2.cvtColor(cv2.imread(str(img_p), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        gt = np.load(gt_p) if gt_p.exists() else np.array([np.nan], np.float32)
        save_dict["img_"+k] = img
        save_dict["gt_"+k] = gt
        SPLIT_DATA[k] = (img, gt if gt.size > 1 else None)
    np.savez_compressed(NPZ_CACHE, **save_dict)
    print(f"Caché empaquetado guardado en Drive: {NPZ_CACHE}  ({len(SPLIT_DATA)} frames)")
    print("En sesiones futuras se cargara de aqui en segundos (no re-extrae).")


In [ ]:
import torch
import numpy as np

sys.path.insert(0, str(EDAM_PATH / "apps" / "depth_estimate"))
from resnet_encoder import ResnetEncoder
from depth_decoder import DepthDecoder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

mono2_encoder = ResnetEncoder(18, False)
enc_dict = torch.load(W_MONO2 / "encoder.pth", map_location=DEVICE)
MONO2_H = enc_dict.get("height", 192)
MONO2_W = enc_dict.get("width",  640)
filtered = {k: v for k, v in enc_dict.items() if k in mono2_encoder.state_dict()}
mono2_encoder.load_state_dict(filtered)
mono2_encoder.to(DEVICE).eval()

mono2_decoder = DepthDecoder(num_ch_enc=mono2_encoder.num_ch_enc, scales=range(4))
mono2_decoder.load_state_dict(torch.load(W_MONO2 / "depth.pth", map_location=DEVICE))
mono2_decoder.to(DEVICE).eval()

print(f"Monodepth2 (SCARED): {MONO2_H}x{MONO2_W} — cargado OK")


In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm", "einops"])

import importlib.util, types
import torch.nn as nn
from collections import OrderedDict

# Los pesos del Dr. Espinosa usan encoder MPViT (MonoViT) pero DOS decoders distintos
# segun el modelo:
#   - MonoViT  -> DepthDecoder simple Monodepth2 (num_ch_dec = num_ch_enc)
#   - MonoIIT  -> HR-Depth (el del paper MonoViT, con convs.f4 / X_00 / attention)
# load_monovit_pair detecta automaticamente cual usar inspeccionando las keys.

_NETDIR = MONOVIT_PATH / "networks"

# --- 1. cargar submodulos del repo MonoViT aislados ---
for _k in list(sys.modules):
    if _k == "networks" or _k.startswith("networks."):
        del sys.modules[_k]
_pkg = types.ModuleType("networks"); _pkg.__path__ = [str(_NETDIR)]
sys.modules["networks"] = _pkg
def _load_sub(name, fname):
    spec = importlib.util.spec_from_file_location(f"networks.{name}", str(_NETDIR / fname))
    m = importlib.util.module_from_spec(spec); sys.modules[f"networks.{name}"] = m
    spec.loader.exec_module(m); setattr(_pkg, name, m); return m

_load_sub("hr_layers", "hr_layers.py")
_hr_dec     = _load_sub("hr_decoder", "hr_decoder.py")
mpvit_small = _load_sub("mpvit", "mpvit.py").mpvit_small
DepthDecoderHR = _hr_dec.DepthDecoder

# --- 2. DepthDecoder simple Monodepth2 (num_ch_dec = num_ch_enc) ---
class _Conv3x3(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.pad = nn.ReflectionPad2d(1)
        self.conv = nn.Conv2d(int(i), int(o), 3)
    def forward(self, x): return self.conv(self.pad(x))
class _ConvBlock(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.conv = _Conv3x3(i, o); self.nonlin = nn.ELU(inplace=True)
    def forward(self, x): return self.nonlin(self.conv(x))
def _up(x): return nn.functional.interpolate(x, scale_factor=2, mode="nearest")

class MonoViTDepthDecoderSimple(nn.Module):
    def __init__(self, num_ch_enc=[64,128,216,288,288], scales=range(4), use_skips=True):
        super().__init__()
        self.scales=list(scales); self.use_skips=use_skips
        self.num_ch_enc=np.array(num_ch_enc); self.num_ch_dec=np.array(num_ch_enc)
        self.convs=OrderedDict()
        for i in range(4,-1,-1):
            ci=self.num_ch_enc[-1] if i==4 else self.num_ch_dec[i+1]
            self.convs[("upconv",i,0)]=_ConvBlock(ci,self.num_ch_dec[i])
            ci=self.num_ch_dec[i]
            if self.use_skips and i>0: ci+=self.num_ch_enc[i-1]
            self.convs[("upconv",i,1)]=_ConvBlock(ci,self.num_ch_dec[i])
        for s in self.scales:
            self.convs[("dispconv",s)]=_Conv3x3(self.num_ch_dec[s],1)
        self.decoder=nn.ModuleList(list(self.convs.values())); self.sigmoid=nn.Sigmoid()
    def forward(self, feats):
        out={}; x=feats[-1]
        for i in range(4,-1,-1):
            x=self.convs[("upconv",i,0)](x); x=[_up(x)]
            if self.use_skips and i>0: x+=[feats[i-1]]
            x=torch.cat(x,1); x=self.convs[("upconv",i,1)](x)
            if i in self.scales: out[("disp",i)]=self.sigmoid(self.convs[("dispconv",i)](x))
        return out

def _build_decoder_for(weights_dir, device):
    """Detecta el tipo de decoder segun las keys del checkpoint."""
    sd = torch.load(weights_dir / "depth.pth", map_location=device)
    is_hr = any(k.startswith("convs.f") or "X_00" in k or "_attention" in k for k in sd)
    if is_hr:
        dec = DepthDecoderHR()
        kind = "HR-Depth"
    else:
        dec = MonoViTDepthDecoderSimple([64,128,216,288,288], scales=range(4))
        kind = "Monodepth2-simple"
    dec.load_state_dict(sd)
    return dec, kind

def load_monovit_pair(weights_dir, device):
    encoder = mpvit_small()
    encoder.num_ch_enc = [64, 128, 216, 288, 288]
    enc_dict = torch.load(weights_dir / "encoder.pth", map_location=device)
    h = enc_dict.get("height", 192); w = enc_dict.get("width", 640)
    encoder.load_state_dict({k: v for k, v in enc_dict.items()
                             if k in encoder.state_dict()})
    encoder.to(device).eval()
    decoder, kind = _build_decoder_for(weights_dir, device)
    decoder.to(device).eval()
    print(f"  decoder: {kind}")
    return encoder, decoder, h, w

# MonoViT real del profe = weights_19_MonoViT (decoder HR-Depth oficial), AbsRel ~0.073.
# monovit_weights (W_MONOVIT) usa decoder simple y da 0.099 -> descartado.
monovit_enc, monovit_dec, MONOVIT_H, MONOVIT_W = load_monovit_pair(W_W19MONO, DEVICE)
print(f"MonoViT (SCARED): {MONOVIT_H}x{MONOVIT_W} — cargado OK")

In [ ]:
import importlib.util

_endosfm_dir = ENDOSLAM_PATH / "EndoSfMLearner"
if str(_endosfm_dir) not in sys.path:
    sys.path.insert(0, str(_endosfm_dir))

_spec = importlib.util.spec_from_file_location(
    "_endosfm_models_a5",
    str(_endosfm_dir / "models" / "__init__.py"),
    submodule_search_locations=[str(_endosfm_dir / "models")]
)
endosfm_models_a5 = importlib.util.module_from_spec(_spec)
sys.modules["_endosfm_models_a5"] = endosfm_models_a5
_spec.loader.exec_module(endosfm_models_a5)

endosfm_scared = endosfm_models_a5.DispResNet(18, False).to(DEVICE)
w = torch.load(ENDOSFM_WEIGHTS, map_location=DEVICE)
endosfm_scared.load_state_dict(w["state_dict"])
endosfm_scared.eval()

n = sum(p.numel() for p in endosfm_scared.parameters())
print(f"EndoSfMLearner (SCARED): {n/1e6:.2f} M — cargado OK")


In [ ]:
# AF-SfMLearner: misma arquitectura que Endo-Depth (ResnetEncoder + DepthDecoder),
# PERO con una diferencia critica en el preprocesamiento del encoder.
#
# El ResnetEncoder de Endo-Depth-and-Motion aplica normalizacion ImageNet
# dentro del forward:  x = (input_image - 0.45) / 0.225
# En el repo oficial de AF-SfMLearner esa misma linea esta COMENTADA
# (networks/resnet_encoder.py), porque sus pesos se entrenaron SIN esa
# normalizacion. Monodepth2 si la espera (por eso Monodepth2 da el valor
# correcto con el encoder de EDAM, pero AF-SfMLearner no).
#
# Solucion minima: subclase que sobrescribe forward() sin la normalizacion,
# usada SOLO para AF-SfMLearner. No afecta a Monodepth2.
class ResnetEncoderNoNorm(ResnetEncoder):
    def forward(self, input_image):
        self.features = []
        x = input_image                       # SIN (x-0.45)/0.225 (igual que AF oficial)
        x = self.encoder.conv1(x)
        x = self.encoder.bn1(x)
        self.features.append(self.encoder.relu(x))
        self.features.append(self.encoder.layer1(self.encoder.maxpool(self.features[-1])))
        self.features.append(self.encoder.layer2(self.features[-1]))
        self.features.append(self.encoder.layer3(self.features[-1]))
        self.features.append(self.encoder.layer4(self.features[-1]))
        return self.features

afsfm_scared_enc = ResnetEncoderNoNorm(18, False)
enc_w = torch.load(W_AFSFM / "encoder.pth", map_location=DEVICE)
AFSFM_H = enc_w.get("height", 256)
AFSFM_W = enc_w.get("width",  320)
filtered = {k: v for k, v in enc_w.items() if k in afsfm_scared_enc.state_dict()}
afsfm_scared_enc.load_state_dict(filtered)
afsfm_scared_enc.to(DEVICE).eval()

afsfm_scared_dec = DepthDecoder(num_ch_enc=afsfm_scared_enc.num_ch_enc, scales=range(4))
afsfm_scared_dec.load_state_dict(torch.load(W_AFSFM / "depth.pth", map_location=DEVICE))
afsfm_scared_dec.to(DEVICE).eval()

print(f"AF-SfMLearner (SCARED): {AFSFM_H}x{AFSFM_W} — cargado OK")

In [ ]:
# MonoIIT: misma arquitectura/código que MonoViT, pesos diferentes
monoIIT_enc, monoIIT_dec, MONOIIT_H, MONOIIT_W = load_monovit_pair(W_MONOIIT, DEVICE)
print(f"MonoIIT (SCARED): {MONOIIT_H}x{MONOIIT_W} — cargado OK")
if (W_MONOIIT / "lighting.pth").exists():
    print("  lighting.pth presente (no usado en inferencia de profundidad)")

In [ ]:
# weights_19_MonoViT: misma estructura, posible MonoIIF
w19_enc, w19_dec, W19_H, W19_W = load_monovit_pair(W_W19MONO, DEVICE)
print(f"weights19-MonoViT (SCARED): {W19_H}x{W19_W} — cargado OK")
if (W_W19MONO / "lighting.pth").exists():
    print("  lighting.pth presente — probable MonoIIF")

In [ ]:
import cv2, numpy as np, torch, torchvision.transforms as T
import importlib.util, types, subprocess

# Dependencias de EndoLMSPEC/EndoViT
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "IQA_pytorch", "path"])

# EndoLMSPEC
def _load_endolmspec(lmspec_path, device):
    _orig = sys.path.copy()
    # Excluir EndoSLAM, HADepth y EndoViT — todos tienen 'utils' que colisiona con utils.pyramids
    clean = [str(lmspec_path)] + [
        p for p in sys.path
        if "EndoSLAM" not in p and "endosfm" not in p.lower()
        and "HADepth" not in p and "EndoViT" not in p and "EndoVit" not in p]
    for k in list(sys.modules):
        if k in ("utils","generator","unet") or k.startswith("utils."): del sys.modules[k]
    try:
        sys.path = clean
        spec = importlib.util.spec_from_file_location("generator", lmspec_path/"generator.py")
        mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        Generator = mod.Generator
    finally:
        sys.path = _orig
    return Generator(n_channels=3, device=device, bilinear=False), Generator

lmspec_net, _ = _load_endolmspec(LMSPEC_PATH, DEVICE)
lmspec_net.load_state_dict(torch.load(LMSPEC_WEIGHTS, map_location=DEVICE))
lmspec_net.to(DEVICE).eval()
print("EndoLMSPEC OK")

# IAT — limpiar utils de EndoLMSPEC antes para que IAT cargue su propio utils
for k in list(sys.modules):
    if k == "utils" or k.startswith("utils."): del sys.modules[k]
sys.modules["imp"] = types.ModuleType("imp")
_iat_model_path = IAT_PATH / "experiments" / "model" / "IAT_main.py"
_spec = importlib.util.spec_from_file_location("IAT_main_a5", _iat_model_path)
_iat_mod = importlib.util.module_from_spec(_spec)
_iat_path = str(IAT_PATH / "experiments")
if _iat_path not in sys.path: sys.path.insert(0, _iat_path)
_spec.loader.exec_module(_iat_mod)
iat_net = _iat_mod.IAT(in_dim=3, with_global=True, type="exp")
iat_net.load_state_dict(torch.load(IAT_WEIGHTS, map_location=DEVICE))
iat_net.to(DEVICE).eval()
print("IAT OK")

def correct_none(img): return img

def correct_retinex(img, sigma=30):
    img_f = img.astype(np.float32) + 1.0
    result = np.zeros_like(img_f)
    for c in range(3):
        blur = cv2.GaussianBlur(img_f[:,:,c],(0,0),sigma)
        result[:,:,c] = np.log(img_f[:,:,c]) - np.log(blur+1.0)
    result -= result.min()
    return (result/(result.max()+1e-8)*255).astype(np.uint8)

def correct_endolmspec(img):
    t = T.ToTensor()(img).to(DEVICE)
    with torch.no_grad(): _, out = lmspec_net(t)
    return (out["subnet_16"][0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

def correct_iat(img):
    t = torch.from_numpy(img.astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): _, _, enh = iat_net(t)
    return (enh[0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

CORRECTIONS = {"none":correct_none,  # "retinex":correct_retinex,  # desactivado a peticion del profesor (conservado por si acaso)
               "endolmspec":correct_endolmspec,"iat":correct_iat}
print(f"Enhancements: {list(CORRECTIONS.keys())}")
# ---------------------------------------------------------------------
# Endo-STTN (Spatio-Temporal Transformer Network) — quita reflejos
# especulares usando informacion temporal de frames vecinos.
# Es TEMPORAL: procesa la secuencia completa de un keyframe de una vez.
# ---------------------------------------------------------------------
import importlib.util as _ilu

# Descargar pesos en Colab si no existen
if IN_COLAB and not STTN_WEIGHTS.exists():
    STTN_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([sys.executable,"-m","pip","install","-q","gdown"])
    import gdown
    gdown.download(id=STTN_GDRIVE_ID, output=str(STTN_WEIGHTS), quiet=False)

def _load_endo_sttn(sttn_path, ckpt, device):
    _orig_path = sys.path.copy()
    _orig_mods = {k: sys.modules[k] for k in list(sys.modules)
                  if k == "core" or k.startswith("core.") or k == "model" or k.startswith("model.")}
    for k in list(sys.modules):
        if k == "core" or k.startswith("core.") or k == "model" or k.startswith("model."):
            del sys.modules[k]
    try:
        sys.path.insert(0, str(sttn_path))
        net_mod = __import__("model.sttn", fromlist=["InpaintGenerator"])
        utils_mod = __import__("core.utils", fromlist=["Stack","ToTorchFormatTensor"])
        model = net_mod.InpaintGenerator().to(device)
        data = torch.load(ckpt, map_location=device)
        model.load_state_dict(data["netG"])
        model.eval()
        Stack = utils_mod.Stack; ToTorch = utils_mod.ToTorchFormatTensor
    finally:
        sys.path = _orig_path
        for k in list(sys.modules):
            if k == "core" or k.startswith("core.") or k == "model" or k.startswith("model."):
                del sys.modules[k]
        sys.modules.update(_orig_mods)
    return model, Stack, ToTorch

STTN_W, STTN_H = 288, 288       # tamaño de entrada del modelo (config oficial)
STTN_REF_LEN, STTN_STRIDE = 10, 5
_sttn_ok = STTN_WEIGHTS.exists()
if _sttn_ok:
    sttn_model, _Stack, _ToTorch = _load_endo_sttn(STTN_PATH, STTN_WEIGHTS, DEVICE)
    from torchvision import transforms as _tvt
    _sttn_to_tensors = _tvt.Compose([_Stack(), _ToTorch()])
    print("Endo-STTN OK")
else:
    print("Endo-STTN: pesos no encontrados, se omitira (subir gen_00009.pth a Drive)")

def _sttn_specular_mask(img_rgb, dil=8):
    """Mascara binaria de reflejos especulares (zonas muy brillantes)."""
    L = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)[:,:,0].astype(np.float32)
    m = (L >= np.percentile(L, 97)).astype(np.uint8)
    if dil:
        m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(dil,dil)), iterations=1)
    return m

def _sttn_get_ref_index(neighbor_ids, length):
    return [i for i in range(0, length, STTN_REF_LEN) if i not in neighbor_ids]

@torch.no_grad()
def endo_sttn_inpaint_sequence(frames_rgb):
    """frames_rgb: lista de np.uint8 HxWx3. Devuelve lista inpainted (mismo tamano original)."""
    if not _sttn_ok:
        return frames_rgb
    H0, W0 = frames_rgb[0].shape[:2]
    pil_frames = [pil.fromarray(f).resize((STTN_W, STTN_H), pil.LANCZOS) for f in frames_rgb]
    masks_np   = [cv2.resize(_sttn_specular_mask(f),(STTN_W,STTN_H),interpolation=cv2.INTER_NEAREST)
                  for f in frames_rgb]
    pil_masks  = [pil.fromarray((m*255).astype(np.uint8)) for m in masks_np]
    vlen = len(pil_frames)

    feats = _sttn_to_tensors(pil_frames).unsqueeze(0)*2-1
    masks = _sttn_to_tensors(pil_masks).unsqueeze(0)
    feats, masks = feats.to(DEVICE), masks.to(DEVICE)
    bin_masks = [np.expand_dims((np.array(m)!=0).astype(np.uint8),2) for m in pil_masks]
    raw_frames = [np.array(f).astype(np.uint8) for f in pil_frames]

    feats = sttn_model.encoder((feats*(1-masks).float()).view(vlen,3,STTN_H,STTN_W))
    _, c, fh, fw = feats.size()
    feats = feats.view(1, vlen, c, fh, fw)

    comp = [None]*vlen
    for f in range(0, vlen, STTN_STRIDE):
        nb_ids = [i for i in range(max(0,f-STTN_STRIDE), min(vlen,f+STTN_STRIDE+1))]
        ref_ids = _sttn_get_ref_index(nb_ids, vlen)
        pred_feat = sttn_model.infer(feats[0, nb_ids+ref_ids], masks[0, nb_ids+ref_ids])
        pred_img = torch.tanh(sttn_model.decoder(pred_feat[:len(nb_ids)]))
        pred_img = ((pred_img+1)/2).cpu().permute(0,2,3,1).numpy()*255
        for i, idx in enumerate(nb_ids):
            # solo rellenar dentro de la mascara; fuera, mantener el original
            img = pred_img[i].astype(np.uint8)*bin_masks[idx] + raw_frames[idx]*(1-bin_masks[idx])
            comp[idx] = img if comp[idx] is None else (comp[idx]*0.5 + img*0.5).astype(np.uint8)
    # volver al tamano original
    return [cv2.resize(c, (W0, H0), interpolation=cv2.INTER_LANCZOS4) for c in comp]

# Cache por keyframe para no re-inpaint la misma secuencia
_STTN_CACHE = {}
def correct_endo_sttn(img_rgb, ds=None, kf=None, fid=None):
    """Si se pasa (ds,kf,fid), usa inpainting temporal de toda la secuencia del keyframe.
    Si no, hace fallback espacial (1 solo frame)."""
    if not _sttn_ok:
        return img_rgb
    if ds is None:
        return endo_sttn_inpaint_sequence([img_rgb])[0]
    key = (ds, kf)
    if key not in _STTN_CACHE:
        fids = SPLIT_BY_KF[key]
        seq = [load_split_frame(ds, kf, f)[0] for f in fids]
        out = endo_sttn_inpaint_sequence(seq)
        _STTN_CACHE.clear()  # mantener memoria baja: solo 1 keyframe a la vez
        _STTN_CACHE[key] = {f: o for f, o in zip(fids, out)}
    return _STTN_CACHE[key][fid]

CORRECTIONS = {"none":correct_none,  # "retinex":correct_retinex,  # desactivado a peticion del profesor (conservado por si acaso)
               "endolmspec":correct_endolmspec,"iat":correct_iat,
               "endosttn":correct_endo_sttn}
print(f"Enhancements: {list(CORRECTIONS.keys())}")


In [ ]:
import time, torch.nn.functional as F, PIL.Image as pil
from torchvision import transforms
from scipy.spatial import cKDTree
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.transform import resize as imresize

FX, FY, CX, CY = 1078.0, 1078.0, 640.0, 512.0

def _predict_monodepth2_style(img_rgb, enc, dec, h, w, device, max_depth=100.0, min_depth=0.1):
    """Monodepth2/AF-SfMLearner: replica exacta de evaluate_depth.py oficial.
    Orden oficial: disp_to_depth -> scaled_disp (a resolucion de red) ->
    cv2.resize de scaled_disp al tamano GT -> pred_depth = 1/scaled_disp.
    (Interpola la DISPARIDAD, no la sigmoide cruda, y NO usa normalizacion ImageNet.)"""
    H, W = img_rgb.shape[:2]
    t = transforms.ToTensor()(pil.fromarray(img_rgb).resize((w,h),pil.LANCZOS)).unsqueeze(0).to(device)
    if device.type=="cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = dec(enc(t))
    if device.type=="cuda": torch.cuda.synchronize()
    ms = (time.perf_counter()-t0)*1000
    # disp_to_depth oficial: scaled_disp = min_disp + (max_disp-min_disp)*sigmoid
    disp = out[("disp",0)].squeeze().cpu().numpy()       # sigmoide cruda a resolucion de red
    min_disp, max_disp = 1.0/max_depth, 1.0/min_depth
    scaled_disp = min_disp + (max_disp - min_disp) * disp
    scaled_disp = cv2.resize(scaled_disp, (W, H))        # resize de la DISPARIDAD al tamano GT
    return 1.0/scaled_disp, ms

def _predict_monovit_style(img_rgb, enc, dec, h, w, device, max_depth=80.0, min_depth=0.1):
    """MonoViT/MonoIIT: encoder mpvit_small + DepthDecoder separados.
    Alineado al evaluate_depth.py oficial de MonoViT: MAX_DEPTH=80, mismo orden que
    monodepth2 (disp_to_depth a resolucion de red -> cv2.resize scaled_disp -> 1/scaled_disp)."""
    H, W = img_rgb.shape[:2]
    t = transforms.ToTensor()(pil.fromarray(img_rgb).resize((w,h),pil.LANCZOS)).unsqueeze(0).to(device)
    if device.type=="cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = dec(enc(t))
    if device.type=="cuda": torch.cuda.synchronize()
    ms = (time.perf_counter()-t0)*1000
    disp = out[("disp",0)].squeeze().cpu().numpy()
    min_disp, max_disp = 1.0/max_depth, 1.0/min_depth
    scaled_disp = min_disp + (max_disp - min_disp) * disp
    scaled_disp = cv2.resize(scaled_disp, (W, H))        # resize DISPARIDAD al tamano GT
    return 1.0/scaled_disp, ms

def predict_mono2(img):     return _predict_monodepth2_style(img, mono2_encoder, mono2_decoder, MONO2_H, MONO2_W, DEVICE)
def predict_monovit(img):   return _predict_monovit_style(img, monovit_enc, monovit_dec, MONOVIT_H, MONOVIT_W, DEVICE)
def predict_afsfm(img):     return _predict_monodepth2_style(img, afsfm_scared_enc, afsfm_scared_dec, AFSFM_H, AFSFM_W, DEVICE, max_depth=150.0, min_depth=0.1)  # AF-SfMLearner: max_depth=150, min_depth=0.1 (options.py)
def predict_monoIIT(img):   return _predict_monovit_style(img, monoIIT_enc, monoIIT_dec, MONOIIT_H, MONOIIT_W, DEVICE)
def predict_w19mono(img):   return _predict_monovit_style(img, w19_enc, w19_dec, W19_H, W19_W, DEVICE)

def predict_endosfm(img):
    H, W = img.shape[:2]
    # EndoSfMLearner (DispResNet) se testea a 256x832 (default oficial: test_disp.py).
    r = imresize(img.astype(np.float32),(256,832)).astype(np.float32)  # float ANTES de resize (skimage normaliza uint8 a 0-1)
    t = torch.from_numpy(((r/255-0.45)/0.225).transpose(2,0,1)).unsqueeze(0).to(DEVICE)
    if DEVICE.type=="cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad(): d = endosfm_scared(t)
    if DEVICE.type=="cuda": torch.cuda.synchronize()
    ms = (time.perf_counter()-t0)*1000
    # test_disp.py oficial: predictions[j] = 1/pred_disp, luego resize en espacio depth
    pred_depth = 1.0 / (d.squeeze().cpu().numpy() + 1e-6)
    return imresize(pred_depth, (H, W)), ms

DEPTH_MODELS = {
    "Monodepth2":      predict_mono2,
    "MonoViT":         predict_monovit,
    "EndoSfMLearner":  predict_endosfm,
    "AF-SfMLearner":   predict_afsfm,
    "MonoIIT":         predict_monoIIT,
}

def specular_mask(img, pct=97, dil=15):
    L = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[:,:,0].astype(np.float32)
    m = (L>=np.percentile(L,pct)).astype(np.uint8)
    return cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(dil,dil))).astype(bool)

def depth_to_pc(d, mask, fx=FX, fy=FY, cx=CX, cy=CY):
    H,W = d.shape; uu,vv = np.meshgrid(np.arange(W),np.arange(H))
    Z = d[mask]
    return np.stack([(uu[mask]-cx)*Z/fx,(vv[mask]-cy)*Z/fy,Z],axis=1)

def chamfer(p1,p2,n=50_000):
    if not len(p1) or not len(p2): return np.nan
    rng = np.random.default_rng(42)
    if len(p1)>n: p1=p1[rng.choice(len(p1),n,replace=False)]
    if len(p2)>n: p2=p2[rng.choice(len(p2),n,replace=False)]
    d1,_ = cKDTree(p2).query(p1); d2,_ = cKDTree(p1).query(p2)
    return float((d1.mean()+d2.mean())/2)

# Chamfer (KD-Trees en CPU) es el cuello de botella del experimento y NO esta
# en la tabla de métricas estándar reportada. Desactivado por defecto: baja ~7h a <1h.
# Ponlo en True solo si necesitas la metrica de nube de puntos 3D.
COMPUTE_CHAMFER = False

def compute_metrics(img_orig, img_corr, depth_rel, gt_mm, cap=150.0):
    valid = (~np.isnan(gt_mm))&(gt_mm>0)&(gt_mm<cap)
    if valid.sum()==0:
        return {k:np.nan for k in ["AbsRel","SqRel","RMSE","RMSELog",
                                   "delta_1","delta_2","delta_3",
                                   "Chamfer","AbsRel_spec","AbsRel_nospec","scale"]}
    scale = np.median(gt_mm[valid])/(np.median(depth_rel[valid])+1e-8)
    pred  = depth_rel*scale; d,gt = pred[valid],gt_mm[valid]
    spec  = specular_mask(img_orig); vs,vn = valid&spec, valid&~spec
    ratio = np.maximum(d/(gt+1e-8), gt/(d+1e-8))
    return {
        "scale":    round(float(scale),4),
        "AbsRel":   round(float(np.mean(np.abs(d-gt)/(gt+1e-8))),4),
        "SqRel":    round(float(np.mean((d-gt)**2/(gt+1e-8))),4),
        "RMSE":     round(float(np.sqrt(np.mean((d-gt)**2))),3),
        "RMSELog":  round(float(np.sqrt(np.mean((np.log(np.clip(d,1e-3,None))-np.log(np.clip(gt,1e-3,None)))**2))),4),
        "delta_1":  round(float(np.mean(ratio<1.25)),4),
        "delta_2":  round(float(np.mean(ratio<1.25**2)),4),
        "delta_3":  round(float(np.mean(ratio<1.25**3)),4),
        "Chamfer":  (round(chamfer(depth_to_pc(np.where(valid,pred,np.nan),valid),
                                   depth_to_pc(np.where(valid,gt_mm,np.nan),valid)),3)
                     if COMPUTE_CHAMFER else np.nan),
        "AbsRel_spec":  round(float(np.mean(np.abs(pred[vs]-gt_mm[vs])/(gt_mm[vs]+1e-8))),4) if vs.sum()>0 else np.nan,
        "AbsRel_nospec":round(float(np.mean(np.abs(pred[vn]-gt_mm[vn])/(gt_mm[vn]+1e-8))),4) if vn.sum()>0 else np.nan,
    }

print(f"Modelos  : {list(DEPTH_MODELS.keys())}")
print(f"Enhanc.  : {list(CORRECTIONS.keys())}")
print(f"Total    : {len(DEPTH_MODELS)*len(CORRECTIONS)*len(SPLIT_ITEMS)} evaluaciones")

In [ ]:
import numpy as np, cv2

def load_split_frame(ds, kf, fid):
    """Carga imagen RGB (uint8) y GT (mm, nan en invalidos) del caché en memoria."""
    img, gt = SPLIT_DATA[_key(ds, kf, fid)]
    return img, gt

# Frames consecutivos del mismo keyframe (para Endo-STTN, que es temporal)
from collections import defaultdict
SPLIT_BY_KF = defaultdict(list)
for ds, kf, fid in SPLIT_ITEMS:
    if _key(ds, kf, fid) in SPLIT_DATA:
        SPLIT_BY_KF[(ds, kf)].append(fid)
for k in SPLIT_BY_KF: SPLIT_BY_KF[k] = sorted(SPLIT_BY_KF[k])

_ds0, _kf0, _fid0 = SPLIT_ITEMS[0]
_img, _gt = load_split_frame(_ds0, _kf0, _fid0)
print(f"Ejemplo {_ds0}/{_kf0} frame {_fid0}: img {_img.shape}  "
      f"GT valido {(~np.isnan(_gt)).mean()*100:.1f}%" if _gt is not None else "sin GT")


## 2. Selección de fotogramas (2 under + 2 over, GT > 30%)

Se eligen los **2 fotogramas más oscuros** y los **2 más brillantes** del split que tengan **más del 30% de cobertura de ground truth** (para que la comparación sea fiable).

In [ ]:
import numpy as np, cv2

CAP_MM = 150.0
MIN_GT_COV = 30.0   # % minimo de cobertura de GT

def _brillo(img): return cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[:,:,0].mean()
def _gt_cov(gt):
    v = np.isfinite(gt) & (gt > 0) & (gt < CAP_MM)
    return float(v.mean()*100.0)

# Caracterizar frames del split con GT suficiente
_chars = []
for ds,kf,fid in SPLIT_ITEMS:
    img,gt = load_split_frame(ds,kf,fid)
    if img is None or gt is None: continue
    cov = _gt_cov(gt)
    if cov < MIN_GT_COV: continue
    _chars.append({"ds":ds,"kf":kf,"fid":fid,"b":_brillo(img),"cov":cov})
_chars.sort(key=lambda r: r["b"])

# Seleccion DIVERSA: 2 under y 2 over de KEYFRAMES DISTINTOS
# (evita elegir frames casi identicos del mismo video, ej. f40 y f35).
def _pick_diverse(cands, n=2):
    out, used_kf = [], set()
    for r in cands:                       # cands ya ordenado por brillo
        key = (r["ds"], r["kf"])
        if key in used_kf: continue
        out.append(r); used_kf.add(key)
        if len(out) == n: break
    # si no hubo suficientes keyframes distintos, completar con lo que haya
    if len(out) < n:
        for r in cands:
            if r not in out:
                out.append(r)
                if len(out) == n: break
    return out

UNDER_CASES = _pick_diverse(_chars, 2)                 # 2 mas oscuros, keyframes distintos
OVER_CASES  = _pick_diverse(list(reversed(_chars)), 2) # 2 mas brillantes, keyframes distintos
ROWS = UNDER_CASES + OVER_CASES          # 4 filas: under, under, over, over

print(f"Frames con GT>{MIN_GT_COV:.0f}%: {len(_chars)}")
print("Filas seleccionadas (keyframes distintos por regimen):")
for r in ROWS:
    print(f"  {r['ds']}/{r['kf']} f{r['fid']}  L={r['b']:.0f}  GT={r['cov']:.0f}%")

## 3. Generar las dos cuadrículas

Para cada enhancement (**IAT**, **Endo-LMSPEC**): 4 filas × (raw + output + 5 depth maps).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy.ma as ma, os

assert "iat" in CORRECTIONS and "endolmspec" in CORRECTIONS, "Faltan enhancements"
GRID_ENH = ["iat", "endolmspec"]   # un grid por cada uno
model_names = list(DEPTH_MODELS.keys())   # 5 modelos en orden

# Disparidad normalizada a [0,1] por frame -> escala comun para la leyenda
def _disp_norm(depth):
    disp = 1.0/np.clip(depth, 1e-6, None)
    hi = np.percentile(disp, 95) + 1e-6
    return np.clip(disp/hi, 0, 1)

os.makedirs(REPO_ROOT/"outcomes"/"avance8", exist_ok=True)
GENERATED_GRIDS = []
_CMAP = "magma"

for enh in GRID_ENH:
    ncol = 2 + len(model_names)          # raw + output + 5 modelos
    nrow = len(ROWS)                     # 4 filas
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.7*ncol + 0.6, 2.9*nrow))
    fig.suptitle(f"Enhancement: {enh.upper()}   |   rows: 2 underexposed + 2 overexposed (GT>{MIN_GT_COV:.0f}%)",
                 fontsize=16, fontweight="bold", y=0.99)

    col_titles = ["input (raw)", f"output ({enh})"] + model_names
    for j, t in enumerate(col_titles):
        axes[0, j].set_title(t, fontsize=12, fontweight="bold")

    _im = None  # referencia para el colorbar
    for i, r in enumerate(ROWS):
        ds,kf,fid = r["ds"],r["kf"],r["fid"]
        img, gt = load_split_frame(ds, kf, fid)
        img_enh = CORRECTIONS[enh](img, ds=ds, kf=kf, fid=fid) if enh=="endosttn" else CORRECTIONS[enh](img)
        reg = "under" if r in UNDER_CASES else "over"
        axes[i,0].set_ylabel(f"{reg}\n{ds}/{kf}\nf{fid} (GT {r['cov']:.0f}%)",
                             fontsize=9, fontweight="bold", rotation=0, ha="right", va="center", labelpad=38)
        axes[i,0].imshow(img);     axes[i,0].set_xticks([]); axes[i,0].set_yticks([])
        axes[i,1].imshow(img_enh); axes[i,1].axis("off")
        for j, mname in enumerate(model_names, start=2):
            depth, _ = DEPTH_MODELS[mname](img_enh)
            _im = axes[i,j].imshow(_disp_norm(depth), cmap=_CMAP, vmin=0, vmax=1)
            axes[i,j].axis("off")

    plt.tight_layout(rect=[0, 0, 0.93, 0.96])

    # --- Leyenda del gradiente (colorbar) a la derecha, como la figura de referencia ---
    cax = fig.add_axes([0.945, 0.12, 0.013, 0.74])   # [left, bottom, width, height]
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0, vmax=1), cmap=_CMAP)
    cb = fig.colorbar(sm, cax=cax)
    cb.set_label("relative disparity  (near ↑ / far ↓)", fontsize=10, fontweight="bold")
    cb.set_ticks([0, 1])
    cb.set_ticklabels(["far", "near"])

    _out = REPO_ROOT/"outcomes"/"avance8"/f"avance8_grid_{enh}.png"
    fig.savefig(_out, dpi=130, bbox_inches="tight")
    GENERATED_GRIDS.append((enh, str(_out)))
    print(f"Guardado: {_out}")
    plt.show()

## 3b. Versiones alternativas — con métricas por celda

Las mismas dos cuadrículas, pero con **AbsRel y RMSE** sobreimpresos en cada mapa de profundidad (calculados contra el ground truth de ese fotograma con *median scaling*). Se guardan con sufijo `_metrics` para **conservar** las versiones limpias.

In [ ]:
import matplotlib as mpl

# Metricas por celda (median scaling vs GT del frame)
def _scaled_metrics(depth, gt):
    valid = np.isfinite(gt) & (gt > 0) & (gt < CAP_MM)
    if valid.sum() == 0: return np.nan, np.nan
    ratio = np.median(gt[valid]) / np.median(depth[valid])
    pred = depth * ratio
    absrel = float(np.mean(np.abs(pred[valid]-gt[valid]) / gt[valid]))
    rmse   = float(np.sqrt(np.mean((pred[valid]-gt[valid])**2)))
    return absrel, rmse

def _disp_norm(depth):
    disp = 1.0/np.clip(depth, 1e-6, None)
    hi = np.percentile(disp, 95) + 1e-6
    return np.clip(disp/hi, 0, 1)

_CMAP = "magma"
GENERATED_GRIDS_METRICS = []

for enh in GRID_ENH:
    ncol = 2 + len(model_names)
    nrow = len(ROWS)
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.7*ncol + 0.6, 2.9*nrow))
    fig.suptitle(f"Enhancement: {enh.upper()}   |   AbsRel / RMSE per cell   |   2 under + 2 over (GT>{MIN_GT_COV:.0f}%)",
                 fontsize=15, fontweight="bold", y=0.99)

    col_titles = ["input (raw)", f"output ({enh})"] + model_names
    for j, t in enumerate(col_titles):
        axes[0, j].set_title(t, fontsize=12, fontweight="bold")

    for i, r in enumerate(ROWS):
        ds,kf,fid = r["ds"],r["kf"],r["fid"]
        img, gt = load_split_frame(ds, kf, fid)
        img_enh = CORRECTIONS[enh](img, ds=ds, kf=kf, fid=fid) if enh=="endosttn" else CORRECTIONS[enh](img)
        reg = "under" if r in UNDER_CASES else "over"
        axes[i,0].set_ylabel(f"{reg}\n{ds}/{kf}\nf{fid} (GT {r['cov']:.0f}%)",
                             fontsize=9, fontweight="bold", rotation=0, ha="right", va="center", labelpad=38)
        axes[i,0].imshow(img);     axes[i,0].set_xticks([]); axes[i,0].set_yticks([])
        axes[i,1].imshow(img_enh); axes[i,1].axis("off")
        for j, mname in enumerate(model_names, start=2):
            depth, _ = DEPTH_MODELS[mname](img_enh)
            axes[i,j].imshow(_disp_norm(depth), cmap=_CMAP, vmin=0, vmax=1); axes[i,j].axis("off")
            ar, rm = _scaled_metrics(depth, gt)
            txt = f"AbsRel {ar:.3f}\nRMSE {rm:.1f}" if np.isfinite(ar) else "n/a"
            axes[i,j].text(0.5, 0.04, txt, transform=axes[i,j].transAxes,
                           fontsize=8.5, fontweight="bold", color="white", ha="center", va="bottom",
                           bbox=dict(boxstyle="round,pad=0.2", fc=(0,0,0,0.55), ec="none"))

    plt.tight_layout(rect=[0, 0, 0.93, 0.96])
    cax = fig.add_axes([0.945, 0.12, 0.013, 0.74])
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0, vmax=1), cmap=_CMAP)
    cb = fig.colorbar(sm, cax=cax)
    cb.set_label("relative disparity  (near ↑ / far ↓)", fontsize=10, fontweight="bold")
    cb.set_ticks([0, 1]); cb.set_ticklabels(["far", "near"])

    _out = REPO_ROOT/"outcomes"/"avance8"/f"avance8_grid_{enh}_metrics.png"
    fig.savefig(_out, dpi=130, bbox_inches="tight")
    GENERATED_GRIDS_METRICS.append((enh, str(_out)))
    print(f"Guardado: {_out}")
    plt.show()

## 4. Lectura

Cada cuadrícula permite comparar, para un mismo enhancement, **cómo responde cada arquitectura** de profundidad bajo sub/sobre-exposición. Las dos primeras filas (under) muestran el comportamiento en penumbra; las dos últimas (over) bajo saturación. El *output* (col. 2) deja ver qué cambió el realce en la imagen antes de la inferencia.